# 03 — Delivery EDA

**Project:** Food Delivery Operations Analytics  
**Author:** Sitanshu Singh

Analyzing delivery performance — late deliveries, peak hours, city differences, and rider efficiency.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

orders = pd.read_csv('../data/cleaned/orders_cleaned.csv', parse_dates=['order_date'])
customers = pd.read_csv('../data/cleaned/customers_cleaned.csv')
riders = pd.read_csv('../data/cleaned/delivery_partners_cleaned.csv')

delivered = orders[orders['status'] == 'Delivered'].copy()
delivered = delivered.merge(customers[['customer_id', 'city']], on='customer_id', how='left')

print(f'Working with {len(delivered)} delivered orders')

## 1. Late Delivery Rate by City

In [ ]:
delivered['is_late'] = delivered['delivery_time_mins'] > 45

city_sla = delivered.groupby('city').agg(
    total_orders=('order_id', 'count'),
    late_orders=('is_late', 'sum'),
    avg_delivery_time=('delivery_time_mins', 'mean')
).reset_index()
city_sla['late_pct'] = city_sla['late_orders'] / city_sla['total_orders'] * 100
city_sla = city_sla.sort_values('late_pct', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#ef4444' if p > 8 else '#f97316' if p > 6 else '#22c55e' for p in city_sla['late_pct']]
bars = ax.barh(city_sla['city'], city_sla['late_pct'], color=colors)
ax.set_xlabel('Late Delivery Rate (%)')
ax.set_title('Late Delivery Rate by City (orders > 45 mins)')
for bar, val in zip(bars, city_sla['late_pct']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center')
plt.tight_layout()
plt.savefig('../reports/late_delivery_by_city.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Peak Hour Analysis

In [ ]:
hourly = delivered.groupby('order_hour').agg(
    orders=('order_id', 'count'),
    late_orders=('is_late', 'sum')
).reset_index()
hourly['late_pct'] = hourly['late_orders'] / hourly['orders'] * 100

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(hourly['order_hour'], hourly['orders'], color='#f97316', alpha=0.7, label='Order Volume')
ax2.plot(hourly['order_hour'], hourly['late_pct'], color='#ef4444', linewidth=2.5, label='Late %', marker='o')

ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Order Volume', color='#f97316')
ax2.set_ylabel('Late Delivery %', color='#ef4444')
ax1.set_title('Order Volume and Late Delivery Rate by Hour')
ax1.set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig('../reports/peak_hour_analysis.png', dpi=150, bbox_inches='tight')
plt.show()